In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
import keras
import seaborn as sn
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import skimage.io
import skimage.transform
import os
import shutil
import random
import joblib
import constants
import time
from joblib import Parallel, delayed

In [48]:
def img_to_512(dir_imgs, dir_labels, new_img_path, new_label_path):
    
    # Remove the directories so we can stat anew
    if os.path.isdir(new_img_path):
        shutil.rmtree(new_img_path)
    os.makedirs(new_img_path)
    if os.path.isdir(new_label_path):
        shutil.rmtree(new_label_path)
    os.makedirs(new_label_path)


    aantal = len(os.listdir(os.path.join(os.getcwd(), dir_imgs)))
    teller_files = 1
    for filename in os.listdir(os.path.join(os.getcwd(), dir_imgs)):
        print(f"{teller_files}/{aantal}", end='\r')
        filename = filename.split(".")[0]
        #Per image
        img = skimage.io.imread(f"{dir_imgs}/{filename}.JPG")
        labels = open(f"{dir_labels}/{filename}.txt")
        lines = labels.readlines()
        lines = [line.rstrip() for line in lines]
        teller = 1
        y = 0
        while(y < constants.Y_END - constants.WIDTH_PIECE): #zijkanten worden weggegooid indien te klein
            x = 0
            while(x < constants.X_END - constants.WIDTH_PIECE): #zijkanten worden weggegooid indien te klein
                x_end = x+constants.WIDTH_PIECE if x+constants.WIDTH_PIECE < constants.X_END else constants.X_END
                y_end = y+constants.WIDTH_PIECE if y+constants.WIDTH_PIECE < constants.Y_END else constants.Y_END
                img_piece = img[y:y_end,x:x_end]
                labels_piece = open(f"{new_label_path}/{filename}_{teller}.txt", "a")
                for line in lines:
                    x_center, y_center, width, height = [round(float(s)) for s in line.split("\t")[1:]]
                    if x_center >= x and x_center < x_end and y_center >= y and y_center < y_end:
                        labels_piece.write("0\t"+str(x_center-x)+"\t"+str(y_center-y)+"\t"+str(width)+"\t"+str(height)+"\n")
                labels_piece.close()
                if os.stat(labels_piece.name).st_size == 0: os.remove(labels_piece.name)
                skimage.io.imsave(f"{new_img_path}/{filename}_{teller}.JPG", img_piece)
                x += constants.WIDTH_PIECE - constants.OVERLAP #+550
                teller += 1
            y += constants.WIDTH_PIECE - constants.OVERLAP
        labels.close()
        teller_files+= 1
    print(f"{teller_files}/{aantal}")

In [49]:
startTime = time.time()
img_to_512("./data/test/", "./data/labels_full_txt/",
           './data/images_512_test/', './data/label_512_test/')
endTime = time.time()-startTime
print(f'Time elapsed: {endTime//60}m {(endTime%60)}s')

101/100
Time elapsed: 3.0m 3.435110330581665s


In [70]:
def img_to_512_new(dir_imgs, dir_labels, new_img_path, new_label_path):
    
    # Remove the directories so we can stat anew
    if os.path.isdir(new_img_path):
        shutil.rmtree(new_img_path)
    os.makedirs(new_img_path)
    if os.path.isdir(new_label_path):
        shutil.rmtree(new_label_path)
    os.makedirs(new_label_path)

    aantal = len(os.listdir(os.path.join(os.getcwd(), dir_imgs)))
    teller_files = 1

    Parallel(n_jobs=-1, prefer='threads', verbose=10)(delayed(split_image_to_512)(filename, dir_imgs, dir_labels, new_img_path, new_label_path) for filename in os.listdir(os.path.join(os.getcwd(), dir_imgs)))
    # for filename in os.listdir(os.path.join(os.getcwd(), dir_imgs)):
    #     split_image_to_512()
    # print(f"{teller_files}/{aantal}")

def split_image_to_512(filename, dir_imgs, dir_labels, new_img_path, new_label_path):
    # print(filename)
    # print(f"{teller_files}/{aantal}", end='\r')
    filename = filename.split(".")[0]
    # Per image
    img = skimage.io.imread(f"{dir_imgs}/{filename}.JPG")
    labels = open(f"{dir_labels}/{filename}.txt")
    lines = labels.readlines()
    lines = [line.rstrip() for line in lines]
    teller = 1
    y = 0
    while(y < constants.Y_END - constants.WIDTH_PIECE): #zijkanten worden weggegooid indien te klein
        x = 0
        while(x < constants.X_END - constants.WIDTH_PIECE): #zijkanten worden weggegooid indien te klein
            x_end = x+constants.WIDTH_PIECE if x+constants.WIDTH_PIECE < constants.X_END else constants.X_END
            y_end = y+constants.WIDTH_PIECE if y+constants.WIDTH_PIECE < constants.Y_END else constants.Y_END
            img_piece = img[y:y_end,x:x_end]
            labels_piece = open(f"{new_label_path}/{filename}_{teller}.txt", "a")
            for line in lines:
                x_center, y_center, width, height = [round(float(s)) for s in line.split("\t")[1:]]
                if x_center >= x and x_center < x_end and y_center >= y and y_center < y_end:
                    labels_piece.write("0\t"+str(x_center-x)+"\t"+str(y_center-y)+"\t"+str(width)+"\t"+str(height)+"\n")
            labels_piece.close()
            if os.stat(labels_piece.name).st_size == 0: os.remove(labels_piece.name)
            skimage.io.imsave(f"{new_img_path}/{filename}_{teller}.JPG", img_piece)
            x += constants.WIDTH_PIECE - constants.OVERLAP #+550
            teller += 1
        y += constants.WIDTH_PIECE - constants.OVERLAP
    labels.close()
    # teller_files+= 1
    # if teller_files == 20:
    #     break

In [69]:
# startTime = time.time()
img_to_512_new("./data/images.full/", "./data/labels_full_txt/",
           './data/images_512/', './data/label_512/')
# endTime = time.time()-startTime
# print(f'Time elapsed: {endTime//60}m {endTime%60}s')

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:   12.2s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   12.7s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:   25.8s
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:   26.8s
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:   39.2s
[Parallel(n_jobs=-1)]: Done  58 tasks      | elapsed:   40.1s
[Parallel(n_jobs=-1)]: Done  72 out of 100 | elapsed:   52.3s remaining:   20.3s
[Parallel(n_jobs=-1)]: Done  83 out of 100 | elapsed:  1.1min remaining:   13.1s
[Parallel(n_jobs=-1)]: Done  94 out of 100 | elapsed:  1.1min remaining:    4.1s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  1.1min finished
